<a href="https://colab.research.google.com/github/anishkr-sahu/Genai/blob/main/Pineconedb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install all the required packages

!pip install langchain
!pip install pinecone-client==2.2.4
!pip install pypdf

In [ ]:
# openai embeddings
!pip install openai
!pip install tiktoken

In [ ]:
#import all the required libraries
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.vectorstores import Pinecone
from langchain.chains import RetrievalQA
from lnagchain.prompts import PromptTemplate
import os

In [ ]:
# load the pdfs
!mkdir pdfs

In [ ]:
# extract the text from the pdfs
loader = PyPDFDirectoryLoader("pdfs")
data = loader.load()

In [ ]:
data

In [ ]:
#split the extracted Data into Text chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
text_chunks = text_splitter.split_documents(data)

In [ ]:
len(text_chunks)
text_chunks[1]

In [ ]:
# settingup openai key
import os
os.environ['openai_api_key'] = ""

In [ ]:
embeddings = OpenAIEmbeddings()
embeddings

In [ ]:
result = embeddings.embed_query("hello")

In [ ]:
len(result)

In [ ]:
# initializing the pinecone
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY','')
PINECONE_API_ENV = os.environ.get('PINECONE_API_ENV','gcp-starter')

In [ ]:
import pinecone
# initialize pinecone
pinecone.init(
    api_key=PINECONE_API_KEY,
    environment=PINECONE_API_ENV
)
index_name = 'test'  # name of your pincone index here

In [ ]:
# create embeddings for each of the text chunk
docsearch = Pinecone.from_documents([t.page_content for t in text_chunks], embeddings,index_name=index_name)


In [ ]:
# if you already have an index, you can load
docsearch = Pinecone.from_existing_index(index_name, embeddings)
docsearch

In [ ]:
#similarity search
query = ""
docs = docsearch.similarity_search(query,k=3)

In [ ]:
# clean form readable

llm = OpenAI()
qa = Retrieval(llm=llm,chain_type="stuff",retriever=docsearch.as_retriever())


In [ ]:
# q/a
query = ""
qa.run(query)

In [ ]:
import sys


In [ ]:
while True:
  user_input = input(f"Input Prompt: ")
  if user_input == 'exit':
    print('Existing')
    sys.exit()
  if user_input == '':
    continue
  result = qa({'query': user_input})
  print(f"answer: {result['result']}")